# 05b_routellm_retrained — RouteLLM-BERT 재학습본 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.1·4.3절. RouteLLM의 BERT 라우터(XLM-RoBERTa-base, 3분류: 강한 모델 승 / 무승부 / 약한 모델 승)를 우리 데이터로 학습한다. 강한 모델 = high 집단, 약한 모델 = low 집단이고, 정답은 집단 풀 확률로 만든 부드러운 확률이다.

- 학습 설정은 공개 체크포인트에 남은 실제 학습 인자(`training_args.bin`)를 따른다: lr 1e-5, weight decay .01, 실효 batch 32, 4 epoch 선형 감소, 500 step마다 검증 손실로 최선 선택, fp32, seed 42. 논문대로 2000 step에서 멈춘다.
- 점수는 원본 `BERTRouter.calculate_strong_win_rate`와 같은 식이며, 원본 함수와 몇 문항에서 직접 대조한다.

**런타임: 2026.07(Python 3.12), T4 GPU.** 예상 약 1~1.5시간(추정). 05a·06과 동시에 돌려도 된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-finetune.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
# torchvision·torchaudio도 torch 2.5.1 짝으로 맞춘다(Colab 기본 timm이 torchvision을 불러온다)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-finetune-colab.txt')], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
SCRIPT = REPO / 'ml/notebooks/router_final/finetune_track.py'
EXTERNAL = Path('/content/ev10-external-finetune')
FINAL = RUN / 'finetune'
METHOD = 'routellm_retrained'
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)

def run_logged(args):
    """Run the script and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(SCRIPT), *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1', TQDM_MININTERVAL='60'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

# 외부 코드: 고정 커밋에서 내려받고 파일별 SHA-256 확인
run_logged(['fetch', '--external-dir', EXTERNAL])

In [ ]:
# 1) 시험 실행: 원천마다 학습 32행, 2 step, 역할마다 점수 40행 (몇 분)
shutil.rmtree('/content/finetune-smoke', ignore_errors=True)
shutil.rmtree('/content/finetune-smoke-work', ignore_errors=True)
run_logged(['run', '--method', METHOD, '--run-dir', RUN, '--external-dir', EXTERNAL, '--output-dir', '/content/finetune-smoke',
            '--work-dir', '/content/finetune-smoke-work', '--smoke', '--device', 'cuda'])
shutil.rmtree('/content/finetune-smoke-work', ignore_errors=True)

In [ ]:
# 2) 본 실행 → Drive finetune/<방법>/<설정>/ 에 설정마다 저장. 체크포인트는 /content(로컬)에만 둔다
run_logged(['run', '--method', METHOD, '--run-dir', RUN, '--external-dir', EXTERNAL, '--output-dir', FINAL,
            '--work-dir', '/content/finetune-work', '--device', 'cuda'])

In [ ]:
# 3) 요약 (런타임을 다시 시작한 뒤에도 이 셀만 실행하면 된다)
from pathlib import Path
import json
FINAL = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001') / 'finetune'
METHOD = 'routellm_retrained'
manifest = json.loads((FINAL / METHOD / 'manifest.json').read_text())
print('complete:', manifest['complete'], '| versions:', manifest['versions'])
for name in manifest['outputs']:
    m = json.loads((FINAL / METHOD / name / 'metrics.json').read_text())
    extra = {k: m[k] for k in ('max_abs_diff_vs_original', 'global_step', 'best_checkpoint') if k in m}
    print(f"{name:45s} fit {m['fit_seconds']:6.0f}s score {m['score_seconds']:5.0f}s cpu {m['cpu_ms_per_question']:6.1f}ms/q "
          f"degenerate={m['degenerate']} {extra}")